# Devoir 1 — Pipeline NLP (Code de la route)

Ce notebook met en œuvre une **chaîne de traitement** du texte juridique marocain (**Code de la route**), à partir du PDF officiel, jusqu'à un **tableau structuré** et un **regroupement thématique** (clustering).

## Contenu du pipeline

1. **Extraction** du texte depuis le PDF (`pdftotext` si disponible, sinon `pypdf` sous Windows).
2. **Normalisation** de l'arabe (tashkeel, hamza, tatweel, marqueurs bidirectionnels, espaces).
3. **Segmentation** par articles (détection des en-têtes *المادة* + numéro).
4. **Extraction d'informations** : amendes (dirhams), points, peines de prison, récidive, catégorie de véhicule, mots-clés, rôle du paragraphe.
5. **Tableau** `pandas` et **statistiques** descriptives.
6. **Représentation TF-IDF** et **clustering K-means** pour explorer des familles d'articles.
7. **Export** CSV (`export_final.csv`).
8. **SpaCy** (optionnel) pour une détection d'entités si le modèle est installé.

### À propos des valeurs `NaN`

Beaucoup d'articles sont des définitions ou des obligations **sans** mention de montant ni de points : une valeur **`NaN` est alors normale**. La cellule sur le *DataFrame* affiche aussi des **exemples d'articles où une sanction est détectée**.


## Installation des paquets

Exécuter une fois (ou après changement d'environnement). Paquets : `pyarabic`, `scikit-learn`, `pypdf`, `spacy`.


In [1]:
!pip install pyarabic scikit-learn pypdf spacy -q


## Imports Python

Bibliothèques pour le texte, l'arabe, le tableau de données et le *machine learning*.


In [2]:
import subprocess
import re
import pandas as pd
import pyarabic.araby as araby
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans


## Extraction du PDF

Fichier attendu par défaut : `code_de_la_route_MA52_05.pdf` (même dossier que le notebook, ou adapter la variable `pdf_path`).


In [3]:
# Extraction du PDF avec fallback compatible Windows
pdf_path = 'code_de_la_route_MA52_05.pdf'
raw_text = ''

try:
    # Option 1: binaire systeme (rapide si disponible)
    result = subprocess.run(['pdftotext', pdf_path, '-'], capture_output=True, check=True)
    raw_text = result.stdout.decode('utf-8', errors='ignore')
except (FileNotFoundError, subprocess.CalledProcessError):
    # Option 2: fallback Python pur (marche sur Windows sans pdftotext)
    from pypdf import PdfReader
    reader = PdfReader(pdf_path)
    raw_text = "\n".join((page.extract_text() or '') for page in reader.pages)

if not raw_text.strip():
    raise ValueError("Extraction PDF vide. Verifie le chemin du fichier ou installe un extracteur OCR.")

print(len(raw_text), "caracteres")
print(raw_text[400:700])


216951 caracteres
2010 مارس25( 1431 ربيع الآخر8 بتاريخ5824ج.ر عدد(
لكتاب الأولا
شروط السير على الطريق العمومية
القسم الأول
رخصة السياقة
الباب الأول
إلزامية رخصة السياقة
1المادة
لا يجوز لأي شخص أن يسوق مركبة ذات محرك أو مجموعة مركبات على الطريق العمومية 
ما لم يكن حاصلا على رخصة للسياقة سارية الصلاحية ومسلمة من قبل ال


## Normalisation du texte arabe

Réduction du bruit (Bidi, voyelles, tirets) tout en **conservant les retours à la ligne** pour la découpe des articles.


In [4]:
def normalize(text):
    # enlever les marqueurs bidi - c'est eux qui causaient le bug du depart
    text = re.sub(r'[\u200b-\u200f\u202a-\u202e\ufeff]', '', text)
    text = araby.strip_tashkeel(text)
    text = araby.normalize_hamza(text)
    text = araby.strip_tatweel(text)

    # normaliser les espaces SANS casser les retours a la ligne
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n\s*\n+', '\n', text)
    return text.strip()

normalized = normalize(raw_text)
print(normalized[:400])


تم ءعداد هذه النسخة من ءجل تسهيل 
مقروءية النص، ولا يحتج ءلا بالنصوص 
في صيغتها المنشورة بالجريدة الرسمية
 المتعلق 52.05القانون رقم 
بمدونة السير على الطرق، 
كما وقع تغييره وتتميمه 
صيغة موطدة بتاريخ 
2024يوليو10
1
الءمانة العامة للحكومة
 52.05القانون رقم 
 1.10.07المتعلق بمدونة السير على الطرق الصادر بتنفيذه الظهير الشريف رقم 
)،2010 فبراير11( 1431 من صفر26بتاريخ 
كما وقع تغييره وتتميمه
)2168)، ص


## Segmentation par articles

Expressions régulières sur les formes *المادة* + numéro (ordre variable selon la mise en page du PDF).


In [5]:
# Le PDF peut contenir soit "المادة 1" soit "1 المادة".
# On gere les deux formes pour fiabiliser la segmentation.
def split_articles(text):
    pattern = re.compile(
        r'(?:^|\n)\s*(?:(?:[\u0627\u0623\u0625\u0671]?لم?اد[\u0629\u0647]\s*(\d+))|(?:(\d+)\s*[\u0627\u0623\u0625\u0671]?لم?اد[\u0629\u0647]))',
        re.UNICODE | re.MULTILINE,
    )
    matches = list(pattern.finditer(text))
    articles = []

    for i, m in enumerate(matches):
        num = m.group(1) or m.group(2)
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        articles.append({'article_id': num, 'texte': text[start:end].strip()})

    return articles

articles = split_articles(normalized)
print(f"{len(articles)} articles trouves")
if articles:
    print(articles[0])
else:
    print("Aucun article detecte. Verifie le format du texte extrait.")


318 articles trouves
{'article_id': '1', 'texte': '1المادة\nلا يجوز لءي شخص ءن يسوق مركبة ذات محرك ءو مجموعة مركبات على الطريق العمومية \nما لم يكن حاصلا على رخصة للسياقة سارية الصلاحية ومسلمة من قبل الءدارة، تناسب صنف \nالمركبة ءو مجموعة المركبات التي يسوقها.'}


## Motifs d'extraction (expressions régulières)

Montants en dirhams (chiffres arabes ou latins, fourchettes, espaces pour les milliers), points de permis, prison ; dictionnaires pour véhicules, mots-clés et rôles.


In [6]:
# Support chiffres arabes/occidentaux + espaces milliers + ponctuation
AR_NUM_MAP = str.maketrans('٠١٢٣٤٥٦٧٨٩', '0123456789')
NUM_RE = r'[0-9\u0660-\u0669][0-9\u0660-\u0669\.,،\s]*'
DHM = r'(?:درهم|دراهم|dirhams?)'

def parse_num(s):
    if not s:
        return None
    s = s.translate(AR_NUM_MAP)
    s = re.sub(r'[\s\u00a0]', '', s)
    s = s.replace('،', '').replace(',', '').replace('.', '')
    m = re.search(r'\d+', s)
    return int(m.group()) if m else None

def is_plausible_fine(n):
    if n is None or n < 50:
        return False
    if 1900 <= n <= 2030:
        return False
    return True

def is_plausible_points(n):
    return n is not None and 1 <= n <= 12

pat_amende_range = re.compile(
    rf'(?:بغرامة\s+)?غرامة\s+من\s*\(?\s*({NUM_RE})\s*\)?\s*(?:الى|إلى|إلي)\s*\(?\s*({NUM_RE})\s*\)?\s*{DHM}',
    re.UNICODE,
)
pat_amende_range2 = re.compile(
    rf'بغرامة\s+من\s*\(?\s*({NUM_RE})\s*\)?\s*(?:الى|إلى|إلي)\s*\(?\s*({NUM_RE})\s*\)?\s*درهم',
    re.UNICODE,
)
pat_amende_between = re.compile(
    rf'بين\s*\(?\s*({NUM_RE})\s*\)?\s*و\s*\(?\s*({NUM_RE})\s*\)?\s*{DHM}',
    re.UNICODE,
)
pat_amende_single = re.compile(
    rf'(?:بغرامة|غرامة|قدرها|أو\s+بغرامة)?\s*\(?\s*({NUM_RE})\s*\)?\s*درهم',
    re.UNICODE,
)
pat_amende_thousands = re.compile(
    rf'\(?\s*({NUM_RE})\s*\)?\s+000\s*{DHM}',
    re.UNICODE,
)
pat_dirham_near = re.compile(
    rf'([0-9\u0660-\u0669][0-9\u0660-\u0669\.,،\s]*?)\s*{DHM}',
    re.UNICODE,
)
pat_points = re.compile(
    rf'(?:خصم|سحب|يخصم)\s+(?:له|منه|منها)?\s*(?:بعدد)?\s*({NUM_RE})\s*(?:نقطة|نقاط|نقط|نقطه)',
    re.UNICODE,
)
pat_points_num_first = re.compile(
    rf'({NUM_RE})\s*(?:نقطة|نقاط|نقط)(?:\s|$|[.,،])',
    re.UNICODE,
)
pat_points_word_first = re.compile(
    rf'(?:نقطة|نقاط|نقط|نقطه)\s*[\(:]\s*({NUM_RE})',
    re.UNICODE,
)
pat_prison = re.compile(r'الحبس\s+من\s+([^.،\n]+?)(?:\s+وبغرامة|\s+أو|[.،\n])', re.UNICODE)

veh = {
    'poids_lourd': re.compile(r'نقل.*?بضائع|شاحنة|يتجاوز.*?3500', re.UNICODE),
    'leger':       re.compile(r'صنف.*?ب|سيارة.*?سياحية', re.UNICODE),
    'moto':        re.compile(r'دراجة.*?نارية|دراجة.*?محرك', re.UNICODE),
}

kws = {
    'vitesse':       re.compile(r'سرعة|تجاوز.*?سرعة', re.UNICODE),
    'alcool':        re.compile(r'كحول|السكر', re.UNICODE),
    'stationnement': re.compile(r'توقف|انتظار|إيقاف', re.UNICODE),
    'telephone':     re.compile(r'هاتف|خلوي', re.UNICODE),
    'ceinture':      re.compile(r'حزام.*?امان', re.UNICODE),
    'nuit':          re.compile(r'ليلا|ليال', re.UNICODE),
    'autoroute':     re.compile(r'طريق.*?سيار', re.UNICODE),
    'accident':      re.compile(r'حادثة|حادث.*?سير', re.UNICODE),
}

roles = {
    'sanction':   re.compile(r'يعاقب|غرامة|الحبس|خصم.*?نقط|سحب.*?نقط', re.UNICODE),
    'obligation': re.compile(r'يجب|يلتزم|لا يجوز', re.UNICODE),
    'definition': re.compile(r'يقصد|يعني|تعريف', re.UNICODE),
}

print('patterns ok')


patterns ok


## Extraction structurée par article

Construction de la liste `records` : une ligne logique par article détecté.


In [7]:
def extract_fine_amounts(t):
    vals = []
    for a, b in pat_amende_range.findall(t):
        na, nb = parse_num(a), parse_num(b)
        if is_plausible_fine(na):
            vals.append(na)
        if is_plausible_fine(nb):
            vals.append(nb)
    for a, b in pat_amende_range2.findall(t):
        na, nb = parse_num(a), parse_num(b)
        if is_plausible_fine(na):
            vals.append(na)
        if is_plausible_fine(nb):
            vals.append(nb)
    for a, b in pat_amende_between.findall(t):
        na, nb = parse_num(a), parse_num(b)
        if is_plausible_fine(na):
            vals.append(na)
        if is_plausible_fine(nb):
            vals.append(nb)
    for x in pat_amende_single.findall(t):
        n = parse_num(x)
        if is_plausible_fine(n):
            vals.append(n)
    for x in pat_amende_thousands.findall(t):
        n = parse_num(x)
        if n is not None:
            n = n * 1000
            if is_plausible_fine(n):
                vals.append(n)
    for m in pat_dirham_near.finditer(t):
        before = t[max(0, m.start() - 80) : m.start()]
        if re.search(r'ج\.ر|الجريدة الرسمية', before):
            continue
        n = parse_num(m.group(1))
        if is_plausible_fine(n):
            vals.append(n)
    return vals


def extract_points(t):
    for pat in (pat_points, pat_points_num_first, pat_points_word_first):
        m = pat.search(t)
        if m:
            n = parse_num(m.group(1))
            if is_plausible_points(n):
                return n
    return None


def extract(art):
    t = art['texte']

    vals = extract_fine_amounts(t)
    vals = sorted(set(vals))
    amende_fixe = vals[0] if len(vals) == 1 else None
    amende_min = min(vals) if vals else None
    amende_max = max(vals) if vals else None
    if len(vals) > 1:
        amende_fixe = None

    points = extract_points(t)

    mpr = pat_prison.search(t)
    prison = mpr.group(1).strip() if mpr else None

    recidive = bool(re.search(r'حالة.*?العود', t))

    cat = None
    for c, p in veh.items():
        if p.search(t):
            cat = c
            break

    tags = [k for k, p in kws.items() if p.search(t)]

    role = 'autre'
    for r, p in roles.items():
        if p.search(t):
            role = r
            break

    desc = re.sub(r'^(?:[\u0627\u0623\u0625\u0671]?لم?اد[\u0629\u0647]\s*\d+|\d+\s*[\u0627\u0623\u0625\u0671]?لم?اد[\u0629\u0647])\s*', '', t).strip()[:200]

    return {
        'article_id':         art['article_id'],
        'infraction_desc':    desc,
        'categorie_vehicule': cat,
        'amende_fixe':        amende_fixe,
        'amende_min':         amende_min,
        'amende_max':         amende_max,
        'points_retrait':     points,
        'peine_prison':       prison,
        'recidive':           recidive,
        'role_paragraphe':    role,
        'mots_cles':          '|'.join(tags) if tags else None,
    }

records = [extract(a) for a in articles]
print(len(records), 'enregistrements')


318 enregistrements


## DataFrame et couverture des champs

Conversion des types numériques, **couverture** (nombre de valeurs renseignées), indicateurs **`sanction_amende`** / **`sanction_points`**, et aperçu des articles avec sanction. Les `NaN` sur les montants signifient souvent *aucune amende chiffrée dans le texte de l'article*.


In [8]:
df = pd.DataFrame(records)
df['article_id']     = pd.to_numeric(df['article_id'], errors='coerce')
df['amende_fixe']    = pd.to_numeric(df['amende_fixe'], errors='coerce')
df['amende_min']     = pd.to_numeric(df['amende_min'], errors='coerce')
df['amende_max']     = pd.to_numeric(df['amende_max'], errors='coerce')
df['points_retrait'] = pd.to_numeric(df['points_retrait'], errors='coerce')

print('coverage:')
print(df[['amende_fixe', 'amende_min', 'amende_max', 'points_retrait', 'peine_prison']].notna().sum())

has_sanction = df['amende_min'].notna() | df['points_retrait'].notna() | df['peine_prison'].notna()
print('\nExemples d articles avec sanction (montant / points / prison):')
print(df.loc[has_sanction, ['article_id', 'amende_min', 'amende_max', 'points_retrait', 'peine_prison']].head(15).to_string())

print('Premiers articles (souvent definitions / obligations sans amende — NaN attendu):')
df.head(10)


coverage:
amende_fixe        5
amende_min         7
amende_max         7
points_retrait     0
peine_prison      19
dtype: int64

Exemples d articles avec sanction (montant / points / prison):
     article_id  amende_min  amende_max  points_retrait                                        peine_prison
125         126         NaN         NaN             NaN                           ثلاثة ءشهر ءلى ثلاث سنوات
126         127         NaN         NaN             NaN                                    شهر ءلى ستة ءشهر
142         143         NaN         NaN             NaN  شهر ءلى ثلاثة ءشهر وبضعف الغرامة المقررة في الفقرة
149         150         NaN         NaN             NaN                                    شهر واحد ءلى ستة
150         151         NaN         NaN             NaN                                                 ستة
151         152      4000.0      8000.0             NaN                                                None
153         154         NaN         NaN             

,article_id,infraction_desc,categorie_vehicule,amende_fixe,amende_min,amende_max,points_retrait,peine_prison,recidive,role_paragraphe,mots_cles
0,1,لا يجوز لءي شخص ءن يسوق مركبة ذات محرك ءو مجمو...,None,NaN,NaN,NaN,NaN,None,False,obligation,None
1,2,استثناء من ءحكام المادة الءولى ءعلاه :\n - يجو...,None,NaN,NaN,NaN,NaN,None,False,autre,None
2,3,يجب على الساءقين الحاصلين على رخصة سياقة مسلمة...,None,NaN,NaN,NaN,NaN,None,False,obligation,None
3,4,في حالة السير الدولي ووفقا للاتفاقية الدولية ل...,None,NaN,NaN,NaN,NaN,None,False,autre,None
4,5,من 13 بتاريخ1.16.106 الصادر بتنفيذه الظهير الش...,leger,NaN,NaN,NaN,NaN,None,False,autre,None
5,6,لا يجوز لءي كان سياقة مركبة فلاحية ذات محرك ءو...,None,NaN,NaN,NaN,NaN,None,False,obligation,None
6,7,من 13 بتاريخ1.16.106 الصادر بتنفيذه الظهير الش...,leger,NaN,NaN,NaN,NaN,None,False,autre,None
7,8,من 13 بتاريخ1.16.106 الصادر بتنفيذه الظهير الش...,leger,NaN,NaN,NaN,NaN,None,False,autre,None
8,9,يجب الءدلاء برخصة السياقة ءو بالوثيقة التي تحل...,None,NaN,NaN,NaN,NaN,None,False,obligation,None
9,10,من 13 بتاريخ1.16.106 الصادر بتنفيذه الظهير الش...,None,NaN,NaN,NaN,NaN,None,False,autre,None


## Statistiques numériques

`describe()` sur les colonnes quantitatives (les indicateurs ne concernent que les lignes où la colonne est renseignée).


In [9]:
df.describe()


,article_id,amende_fixe,amende_min,amende_max,points_retrait
count,318.000000,5.000000,7.000000,7.000000,0.0
mean,159.500000,134287.400000,96696.285714,107062.428571,NaN
std,91.942917,153896.504164,141108.072201,135172.432895,NaN
min,1.000000,1437.000000,1437.000000,1437.000000,NaN
25%,80.250000,70000.000000,2718.500000,39000.000000,NaN
50%,159.500000,100000.000000,70000.000000,70000.000000,NaN
75%,238.750000,100000.000000,100000.000000,100000.000000,NaN
max,318.000000,400000.000000,400000.000000,400000.000000,NaN


## Vecteurs TF-IDF

Corpus = texte intégral de chaque article ; matrice creuse pour le clustering.


In [10]:
corpus = [a['texte'] for a in articles]

vec = TfidfVectorizer(analyzer='word', min_df=2, max_df=0.85, sublinear_tf=True)
X = vec.fit_transform(corpus)
print(X.shape)


(318, 2462)


## Clustering K-means

Attribution d'un `cluster_id` à chaque article ; affichage des termes les plus centraux par groupe.


In [11]:
km = KMeans(n_clusters=8, random_state=42, n_init=10)
km.fit(X)

df['cluster_id'] = km.labels_

features = vec.get_feature_names_out()
for i in range(8):
    top = km.cluster_centers_[i].argsort()[::-1][:5]
    mots = [features[j] for j in top]
    print(f"cluster {i}: {mots}")


cluster 0: ['بالمركبات', 'تحدث', 'تتعلق', 'برخص', 'المادتين']
cluster 1: ['من', 'هذا', 'القانون', 'في', 'ءو']
cluster 2: ['على', 'في', 'الءدارة', 'رخصة', 'السياقة']
cluster 3: ['درهم', 'بغرامة', '000', 'ءلى', 'العود']
cluster 4: ['العمومية', 'الطريق', 'على', 'المركبات', 'ءو']
cluster 5: ['درهم', 'ءلى', 'من', 'ءو', 'بالحبس']
cluster 6: ['ءو', 'في', 'المركبة', 'على', 'ءن']
cluster 7: ['من', '1437', 'رقم', 'الظهير', 'بتاريخ1']


## Export CSV

Sauvegarde en `utf-8-sig` pour une ouverture correcte sous Excel (Windows).


In [12]:
df.to_csv("export_final.csv", index=False, encoding='utf-8-sig', na_rep='')
print("done ->", df.shape)
df.head()


done -> (318, 12)


,article_id,infraction_desc,categorie_vehicule,amende_fixe,amende_min,amende_max,points_retrait,peine_prison,recidive,role_paragraphe,mots_cles,cluster_id
0,1,لا يجوز لءي شخص ءن يسوق مركبة ذات محرك ءو مجمو...,None,NaN,NaN,NaN,NaN,None,False,obligation,None,4
1,2,استثناء من ءحكام المادة الءولى ءعلاه :\n - يجو...,None,NaN,NaN,NaN,NaN,None,False,autre,None,1
2,3,يجب على الساءقين الحاصلين على رخصة سياقة مسلمة...,None,NaN,NaN,NaN,NaN,None,False,obligation,None,6
3,4,في حالة السير الدولي ووفقا للاتفاقية الدولية ل...,None,NaN,NaN,NaN,NaN,None,False,autre,None,1
4,5,من 13 بتاريخ1.16.106 الصادر بتنفيذه الظهير الش...,leger,NaN,NaN,NaN,NaN,None,False,autre,None,7


## SpaCy (optionnel)

Si `spacy` et le modèle multilingue sont disponibles, test sur les premières entités ; sinon, affichage des champs extraits par expressions régulières.


In [13]:
try:
    import spacy  # type: ignore[import-untyped]
    try:
        nlp = spacy.load('xx_ent_wiki_sm')
    except Exception:
        # Fallback sans NER si le modele n'est pas installe
        nlp = spacy.blank('xx')

    for art in articles[:3]:
        doc = nlp(art['texte'][:300])
        ents = [(e.text, e.label_) for e in getattr(doc, 'ents', [])]
        print(f"article {art['article_id']}:", ents)
except Exception as e:
    print('spacy pas dispo:', e)
    for r in records[:5]:
        print(f"art {r['article_id']} | amende={r['amende_fixe']} | points={r['points_retrait']} | tags={r['mots_cles']}")


spacy pas dispo: No module named 'spacy'
art 1 | amende=None | points=None | tags=None
art 2 | amende=None | points=None | tags=None
art 3 | amende=None | points=None | tags=None
art 4 | amende=None | points=None | tags=None
art 5 | amende=None | points=None | tags=None
